<a href="https://colab.research.google.com/github/jeevana-049/MABSA/blob/main/MABSA_step1_caption_ocr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch pillow requests

In [2]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import requests

# Load the model that describes images (called BLIP)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Download a sample photo from the internet
url = "https://images.unsplash.com/photo-1516035069371-29a1b244cc32?w=600"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

# Ask the model to describe it
inputs = processor(image, return_tensors="pt")
output = model.generate(**inputs)
caption = processor.decode(output[0], skip_special_tokens=True)

print("Caption:", caption)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1954: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Caption: a camera and some lenses on a table


In [3]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 31.7 MB/s eta 0:00:00


In [6]:
import easyocr
from PIL import Image, ImageDraw, ImageFont

# Make a white picture and write some words on it
img = Image.new("RGB", (600, 200), "white")
draw = ImageDraw.Draw(img)
font = ImageFont.load_default(size=40)
draw.text((30, 70), "BATTERY LOW 10%", fill="black", font=font)
img.save("sample.png")

# Load the text reader (English)
reader = easyocr.Reader(['en'])

# Read the text
results = reader.readtext("sample.png")

for box, text, confidence in results:
    print("Text found:", text, "| Confidence:", round(confidence, 2))

Text found: BATTERY LOW 10% | Confidence: 0.9


In [7]:
from transformers import pipeline
from PIL import Image
import re

# Sentiment model (gives positive / neutral / negative)
sentiment = pipeline("sentiment-analysis",
                     model="cardiffnlp/twitter-roberta-base-sentiment-latest")

def get_caption(image_path):
    img = Image.open(image_path).convert("RGB")
    inputs = processor(img, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=30)
    return processor.decode(out[0], skip_special_tokens=True)

def get_ocr_text(image_path):
    results = reader.readtext(image_path)
    return " ".join(text for _, text, _ in results)

def filter_by_aspect(text, aspect):
    # Split into small parts and keep only parts that mention the aspect
    parts = re.split(r"[.,;!?]| but | and ", text)
    kept = [p.strip() for p in parts if aspect.lower() in p.lower()]
    return ". ".join(kept) if kept else text

def analyze(text=None, image_path=None, aspect=None):
    pieces = []

    if text:
        pieces.append(text)
    if image_path:
        caption = get_caption(image_path)
        ocr = get_ocr_text(image_path)
        print("Caption:", caption)
        print("OCR text:", ocr if ocr else "(none)")
        pieces.append(caption)
        if ocr:
            pieces.append(ocr)

    if not pieces:
        return "Please give a text, an image, or both."

    mode = "text + image" if (text and image_path) else ("text only" if text else "image only")
    combined = ". ".join(pieces)

    if aspect:
        combined = filter_by_aspect(combined, aspect)

    result = sentiment(combined)[0]
    print(f"Mode: {mode}")
    print(f"Aspect: {aspect if aspect else 'overall'}")
    print(f"Used text: {combined}")
    print(f"Sentiment: {result['label']} ({round(result['score'], 2)})")

# 1) Text only
analyze(text="Great camera, but the delivery was slow.", aspect="camera")
print("-----")
analyze(text="Great camera, but the delivery was slow.", aspect="delivery")
print("-----")

# 2) Image only (uses the sample.png you made in the OCR step)
analyze(image_path="sample.png")
print("-----")

# 3) Both
analyze(text="The battery is terrible.", image_path="sample.png", aspect="battery")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Mode: text only
Aspect: camera
Used text: Great camera
Sentiment: positive (0.92)
-----
Mode: text only
Aspect: delivery
Used text: the delivery was slow
Sentiment: negative (0.85)
-----
Caption: battery low 10 %
OCR text: BATTERY LOW 10%
Mode: image only
Aspect: overall
Used text: battery low 10 %. BATTERY LOW 10%
Sentiment: neutral (0.62)
-----
Caption: battery low 10 %
OCR text: BATTERY LOW 10%
Mode: text + image
Aspect: battery
Used text: The battery is terrible. battery low 10 %. BATTERY LOW 10%
Sentiment: negative (0.95)
